# Day 21: Sentiment Analysis Mini-Project

**Week 3 - Text Classification with Classical ML + Practical Workflow**

Welcome to Day 21! Today you'll build an end-to-end **Sentiment Analysis** pipeline on a real dataset. You'll practice text preprocessing, feature extraction with TF‑IDF, baseline modeling with classical ML, evaluation with multiple metrics, and simple tuning.

---

## Learning Objectives

By the end of this project, you will be able to:

1. **Prepare raw text** via minimal, effective preprocessing
2. **Vectorize text** using Bag‑of‑Words / TF‑IDF
3. **Train baseline classifiers** (Logistic Regression, Linear SVM, Naive Bayes)
4. **Evaluate models** with accuracy, precision, recall, F1
5. **Perform light tuning** with GridSearchCV
6. **Analyze errors** and reflect on next steps

### Key Concepts Practiced

- Text cleaning and tokenization (practical, not over-engineered)
- Train/validation/test split and leakage prevention
- Feature extraction with `TfidfVectorizer`
- Baseline ML for NLP: LR, LinearSVC, MultinomialNB
- Metrics beyond accuracy: class balance awareness
- Simple error analysis to guide improvements

---



## Project Overview

### The Challenge

Build a sentiment classifier on a real-world text dataset using a clear, reproducible pipeline.

- **Task**: Binary sentiment classification (positive vs. negative)
- **Dataset**: Kaggle — [Sentiment Analysis Dataset](https://www.kaggle.com/datasets/mgmitesh/sentiment-analysis-dataset)
- **Objective**: Train, evaluate, and compare baseline text classifiers; reflect on improvements

### Why This Matters

- Text classification powers reviews, support tickets, and social media analytics
- Strong baselines (LR/SVM/NB) are fast, often competitive, and easy to deploy
- Clean, minimal pipelines are easier to maintain and extend

### What You Will Build

- Minimal text preprocessing pipeline
- TF‑IDF features with n‑grams
- Baseline models + evaluation and comparison
- Short error analysis to guide next steps

---



## Part 1: Setup and Data Loading

Follow these steps to get your data ready:

1. Download the dataset from Kaggle: [Sentiment Analysis Dataset](https://www.kaggle.com/datasets/mgmitesh/sentiment-analysis-dataset)
2. Place the CSV file in the `data/` folder (create it if needed), or set an absolute path
3. Identify the correct column names for text and label (common: `text`, `review`, `tweet` and `label`, `sentiment`, `target`)

I'll write simple code fro you to load the data and inspect it.

---


In [ ]:
# TODO: Import required libraries
# Core
import os
import re
import string
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Model & Metrics
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

# Text features
from sklearn.feature_extraction.text import TfidfVectorizer

# Baseline models
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

# Plot style
sns.set(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 5)

print("Libraries imported.")


In [4]:
# Configuration: set your dataset path and column names
# Hint: Change DATA_PATH to your actual CSV path if needed

# Dowload the dataset from kaggle: https://www.kaggle.com/datasets/mgmitesh/sentiment-analysis-dataset

DATA_PATH = os.path.join("data", "sentiment_dataset.csv")  # e.g., data/sentiment_dataset.csv
TEXT_COLUMN = "text"      # e.g., "text", "review", "tweet", "message"
LABEL_COLUMN = "label"    # e.g., "label", "sentiment", "target", "polarity"
RANDOM_STATE = 42
TEST_SIZE = 0.2

print("DATA_PATH:", DATA_PATH)
print("TEXT_COLUMN:", TEXT_COLUMN, "| LABEL_COLUMN:", LABEL_COLUMN)


DATA_PATH: data\sentiment_dataset.csv
TEXT_COLUMN: text | LABEL_COLUMN: label


In [ ]:
# Load dataset
# TODO: Update DATA_PATH/TEXT_COLUMN/LABEL_COLUMN if needed

assert os.path.exists(DATA_PATH), f"File not found: {DATA_PATH}. Please set DATA_PATH correctly."

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
print("Columns:", list(df.columns))

# Quick sanity checks
assert TEXT_COLUMN in df.columns, f"TEXT_COLUMN '{TEXT_COLUMN}' not found in columns."
assert LABEL_COLUMN in df.columns, f"LABEL_COLUMN '{LABEL_COLUMN}' not found in columns."

# Drop rows with missing values in essential columns
df = df[[TEXT_COLUMN, LABEL_COLUMN]].dropna()

# Standardize labels to integers if needed (0/1)
if df[LABEL_COLUMN].dtype == "O":
    # Simple mapping: assume two unique string labels
    unique_labels = sorted(df[LABEL_COLUMN].unique())
    label_map = {lbl: idx for idx, lbl in enumerate(unique_labels)}
    df[LABEL_COLUMN] = df[LABEL_COLUMN].map(label_map)
    print("Label mapping:", label_map)

print(df.head())
print(df[LABEL_COLUMN].value_counts(normalize=True).rename("class_ratio"))


**Reflection Question 1:** What is the class balance? If it's imbalanced, which metrics will you prioritize and why?

*Write your observations here:*


---

## Part 2: Quick EDA (Text)

Before modeling, look at basic characteristics:

- Class distribution
- Text length distribution (characters/words)
- Few example samples per class



In [ ]:
# Class distribution plot
class_counts = df[LABEL_COLUMN].value_counts().sort_index()
ax = class_counts.plot(kind='bar', color=["#4a90e2", "#7fb3ff"])  # cool blue tones
ax.set_title("Class Distribution")
ax.set_xlabel("Class")
ax.set_ylabel("Count")
plt.show()

# Text length distributions
df["char_len"] = df[TEXT_COLUMN].astype(str).str.len()
df["word_len"] = df[TEXT_COLUMN].astype(str).str.split().apply(len)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["char_len"], bins=40, ax=axes[0], color="#4a90e2")
axes[0].set_title("Character Length Distribution")
sns.histplot(df["word_len"], bins=40, ax=axes[1], color="#7fb3ff")
axes[1].set_title("Word Count Distribution")
plt.show()

# Show a few samples per class
for cls in sorted(df[LABEL_COLUMN].unique()):
    print(f"\nSamples for class {cls}:")
    display(df[df[LABEL_COLUMN] == cls].head(3))


**Reflection Question 2:** What do you notice about text lengths? Any outliers or patterns that could affect modeling?

*Write your insights here:*


---

## Part 3: Minimal Text Preprocessing

We will keep preprocessing simple and practical:
- Lowercasing
- Remove URLs, mentions, and extra whitespace
- Optional: remove punctuation/numbers

Avoid heavy normalization at first (stemming/lemmatization). Start simple, measure impact, iterate.



In [ ]:
# TODO: Implement a minimal clean_text function
# Hint: Use regex to strip URLs, mentions, and non-letter chars if desired

url_pattern = re.compile(r"https?://\S+|www\.\S+")
mention_pattern = re.compile(r"@[A-Za-z0-9_]+")
non_letter_pattern = re.compile(r"[^a-zA-Z\s]")


def clean_text(text: str, keep_punct: bool = False) -> str:
    if not isinstance(text, str):
        return ""
    text = text.lower().strip()
    text = url_pattern.sub(" ", text)
    text = mention_pattern.sub(" ", text)
    text = re.sub(r"\s+", " ", text)
    if not keep_punct:
        text = non_letter_pattern.sub(" ", text)
        text = re.sub(r"\s+", " ", text)
    return text.strip()

# Apply minimal cleaning
df["text_clean"] = df[TEXT_COLUMN].apply(lambda t: clean_text(t, keep_punct=False))
print(df[[TEXT_COLUMN, "text_clean"]].head(3))


**Reflection Question 3:** Which minimal cleaning steps improved readability without destroying useful signals (e.g., emojis, punctuation)?

*Explain briefly:*


---

## Part 4: Split and Vectorize (TF‑IDF)

We will split the data and extract features using `TfidfVectorizer`:
- Use word n‑grams (e.g., 1–2) to capture short phrases
- Limit max features to keep it simple and fast
- Do not fit on test data (prevent leakage)



In [ ]:
# Train/validation split
X = df["text_clean"].values
y = df[LABEL_COLUMN].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

print("Train size:", len(X_train), "| Test size:", len(X_test))

# Vectorizer
# TODO: Try different ngram_range (1,1), (1,2), max_features like 10k, 50k
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=20000,
    min_df=2,
    max_df=0.9,
    sublinear_tf=True
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print("X_train_vec shape:", X_train_vec.shape)
print("X_test_vec shape:", X_test_vec.shape)


**Reflection Question 4:** Why must the vectorizer be fit only on training data? What kind of leakage happens otherwise?

*Explain in 2–3 sentences:*


---

## Part 5: Baseline Models and Training

We'll start with three strong baselines for text classification:
- Logistic Regression
- Linear SVM (`LinearSVC`)
- Multinomial Naive Bayes



In [ ]:
# TODO: Define baseline models (minimal, sensible defaults)
models = {
    "LogReg": LogisticRegression(max_iter=200, C=2.0, n_jobs=None),
    "LinearSVC": LinearSVC(C=1.0),
    "MultinomialNB": MultinomialNB(alpha=0.5)
}

print("Models:", list(models.keys()))

# Train and evaluate
results = []
for name, clf in models.items():
    clf.fit(X_train_vec, y_train)
    y_pred = clf.predict(X_test_vec)

    acc = accuracy_score(y_test, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, average="binary", zero_division=0)

    results.append({
        "model": name,
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1
    })

results_df = pd.DataFrame(results).sort_values(by="f1", ascending=False)
results_df


In [ ]:
# Confusion matrix and classification report for the best model
best_model_name = results_df.iloc[0]["model"]
best_clf = models[best_model_name]

print("Best model:", best_model_name)

y_pred_best = best_clf.predict(X_test_vec)
print(classification_report(y_test, y_pred_best, digits=4))

cm = confusion_matrix(y_test, y_pred_best)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title(f"Confusion Matrix: {best_model_name}")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()


**Reflection Question 5:** Which model performed best and why? If class imbalance exists, which metric do you trust most?

*Write your analysis here:*


---

## Part 6: Light Hyperparameter Tuning (Optional but Recommended)

Try a quick grid search on your top model. Keep grids small and focused.



In [ ]:
# TODO: Example tuning for Logistic Regression
# Hint: Reduce grid size if your dataset is large. Use scoring='f1' for imbalanced data.

param_grid = {
    "C": [0.5, 1.0, 2.0, 4.0],
}

logreg = LogisticRegression(max_iter=300)

# Using a small CV for speed; increase for more robust estimates
clf = GridSearchCV(logreg, param_grid=param_grid, scoring="f1", cv=3, n_jobs=-1)
clf.fit(X_train_vec, y_train)

print("Best params:", clf.best_params_)
print("Best CV f1:", clf.best_score_)

# Evaluate on test
best_lr = clf.best_estimator_
y_pred_lr = best_lr.predict(X_test_vec)
print("Test report (best LR):\n", classification_report(y_test, y_pred_lr, digits=4))


**Reflection Question 6:** Did tuning materially improve performance? If not, which modeling or feature step would you try next?

*Write your plan here:*


---

## Part 7: Simple Error Analysis

Look at misclassified examples to see patterns. This often reveals data or feature issues.



In [ ]:
# Inspect a few false positives/negatives for the best model
pred = best_clf.predict(X_test_vec)
mask_fp = (pred == 1) & (y_test == 0)
mask_fn = (pred == 0) & (y_test == 1)

print("\nFalse Positives (pred=1, true=0):")
for i, idx in enumerate(np.where(mask_fp)[0][:5]):
    print(f"[{i}]", X_test[idx][:200])

print("\nFalse Negatives (pred=0, true=1):")
for i, idx in enumerate(np.where(mask_fn)[0][:5]):
    print(f"[{i}]", X_test[idx][:200])


**Reflection Question 7:** What common traits do misclassified texts share? How might you adjust preprocessing or features to fix them?

*Note down 2–3 actionable ideas:*


---

## Next Steps (Explore and Improve)

Try these targeted improvements:

- Feature engineering: try `ngram_range=(1,3)`, increase `max_features`, adjust `min_df/max_df`
- Stopwords: pass a stopword list to the vectorizer and compare
- Class weights: use `class_weight='balanced'` for LR/SVM if classes are imbalanced
- Alternative vectorizers: `CountVectorizer` or character n‑grams
- Tuning: grid search `C` for LR/SVM; `alpha` for NB
- Data cleaning: handle emojis, repeated characters, negations ("not good")

---


## Project Submission Guidelines

### Submission Requirements

Before submitting your project, ensure you have:

1. **Completed all TODO sections** with working code
2. **Answered all reflection questions** with detailed analysis
3. **Generated all required visualizations** 
4. **Tested code and verified** all outputs
5. **Documented findings** and insights

### How to Submit

**Step 1:** Send LinkedIn connection request to: https://www.linkedin.com/in/hashirahmed07/

**Step 2:** Clean up notebook and ensure all cells run without errors

**Step 3:** Submit with title format: **30_Days_ML_Practice_Project_Week2**

### Evaluation Criteria

Your project will be evaluated based on:

- **Code Quality (30%)** - Correctness, organization, readability.
- **Analysis Quality (30%)** - Depth of EDA and interpretation.
- **Visualization (20%)** - Clarity and professionalism.
- **Insights (20%)** - Model comparison and justification.

---

**Good luck with your project! Remember: The goal is to learn and practice, not just to get perfect results.**
